# Employee Data Cleaning — Missing Values, Duplicates, Data Types & Inconsistent Values

This notebook cleans a messy public-style HR dataset using **pandas**.

**Issues fixed:** missing values, duplicate records, incorrect data types, inconsistent categorical values.

## 1. Load the raw data

In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('data/raw_data.csv')
print(f'Raw shape: {df.shape}')
df.head()

Raw shape: (69, 11)


,EmployeeID,Full Name,Age,Gender,Department,City,Join_Date,Salary,Email,Rating,Active
0,E1018,Divya Rao,36.0,FEMALE,hr,Jaipur,2022-07-19,97692.0,divya.rao14@yahoo.com,5,No
1,E1054,Neha Malhotra,23.0,MALE,Human Resources,Jaipur,02/11/2020,$29852,neha.malhotra76@company.com,2,0
2,E1012,V. Das,twenty-five,MALE,sales,chennai,02/11/2020,"32,100",NaN,4.5,No
3,E1048,SURESH CHOPRA,NaN,m,HR,Kolkata,02/11/2020,NaN,suresh.chopra86@gmail.com,3,N
4,E1028,Sonal Malhotra,unknown,MALE,Marketing,Mumbai,14/05/2021,₹80724,sonal.malhotra90@GMAIL.COM,NaN,Yes


## 2. First look — spot the problems

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   EmployeeID  69 non-null     str  
 1   Full Name   69 non-null     str  
 2   Age         51 non-null     str  
 3   Gender      68 non-null     str  
 4   Department  66 non-null     str  
 5   City        67 non-null     str  
 6   Join_Date   65 non-null     str  
 7   Salary      58 non-null     str  
 8   Email       62 non-null     str  
 9   Rating      55 non-null     str  
 10  Active      64 non-null     str  
dtypes: str(11)
memory usage: 6.1 KB


In [3]:
df.isna().sum()  # note: this UNDERCOUNTS missing values, since some are hidden as 'N/A' / 'unknown' text

EmployeeID     0
Full Name      0
Age           18
Gender         1
Department     3
City           2
Join_Date      4
Salary        11
Email          7
Rating        14
Active         5
dtype: int64

In [4]:
df.duplicated().sum()  # exact duplicate rows

np.int64(7)

In [5]:
df['Gender'].unique()  # inconsistent spellings

<StringArray>
['FEMALE', 'MALE', 'm', 'Woman', 'Male', 'M', 'f', 'Female', 'F', 'Man', nan]
Length: 11, dtype: str

## 3. Standardize hidden missing-value markers
`"N/A"`, `"unknown"`, and blanks all mean the same thing — convert them all to real `NaN`.

In [6]:
MISSING_TOKENS = ['N/A', 'n/a', 'unknown', 'Unknown', '', ' ']
df.replace(MISSING_TOKENS, np.nan, inplace=True)

,EmployeeID,Full Name,Age,Gender,Department,City,Join_Date,Salary,Email,Rating,Active
0,E1018,Divya Rao,36.0,FEMALE,hr,Jaipur,2022-07-19,97692.0,divya.rao14@yahoo.com,5,No
1,E1054,Neha Malhotra,23.0,MALE,Human Resources,Jaipur,02/11/2020,$29852,neha.malhotra76@company.com,2,0
2,E1012,V. Das,twenty-five,MALE,sales,chennai,02/11/2020,"32,100",NaN,4.5,No
3,E1048,SURESH CHOPRA,NaN,m,HR,Kolkata,02/11/2020,NaN,suresh.chopra86@gmail.com,3,N
4,E1028,Sonal Malhotra,NaN,MALE,Marketing,Mumbai,14/05/2021,₹80724,sonal.malhotra90@GMAIL.COM,NaN,Yes
...,...,...,...,...,...,...,...,...,...,...,...
64,E1038,V. Nair,38.0,F,Finance,Ahmedabad,2022-07-19,74674,NaN,4.5,yes
65,E1009,S. Rao,NaN,M,IT,pune,"May 14, 2021","99,364",suresh.rao63@gmail.com,five,yes
66,E1033,Manish Kumar,-26,FEMALE,marketing,bangalore,14-May-2021,74692.0,manish.kumar54@company.com,2,0
67,E1022,tanya rao,NaN,Female,Operations,Hyderabad,02/11/2020,₹54154,tanya.rao59@company.com,5,no


## 4. Clean Full Name — trim whitespace, standardize case

In [7]:
df['Full Name'] = (
    df['Full Name'].astype(str).str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.title()
)

## 5. Clean Age — word numbers, units, impossible values

In [8]:
WORD_TO_NUM = {"twenty-five": 25, "thirty": 30, "forty": 40, "fifty": 50}

def clean_age(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().lower()
    if val in WORD_TO_NUM:
        return WORD_TO_NUM[val]
    val = val.replace("yrs", "").strip()
    try:
        num = float(val)
    except ValueError:
        return np.nan
    if num < 0 or num > 100:
        return np.nan
    return num

df['Age'] = df['Age'].apply(clean_age)
df['Age'].describe()

count    39.000000
mean     38.256410
std      11.273439
min      22.000000
25%      28.500000
50%      37.000000
75%      49.500000
max      57.000000
Name: Age, dtype: float64

## 6. Standardize Gender

In [9]:
GENDER_MAP = {"male": "Male", "m": "Male", "man": "Male",
              "female": "Female", "f": "Female", "woman": "Female"}
df['Gender'] = df['Gender'].astype(str).str.strip().str.lower().map(GENDER_MAP)
df['Gender'].value_counts()

Gender
Male      39
Female    29
Name: count, dtype: int64

## 7. Standardize Department

In [10]:
DEPT_MAP = {"hr": "HR", "human resources": "HR", "sales": "Sales",
            "marketing": "Marketing", "operations": "Operations", "ops": "Operations",
            "finance": "Finance", "it": "IT", "it dept": "IT"}
df['Department'] = df['Department'].astype(str).str.strip().str.lower().map(DEPT_MAP)
df['Department'].value_counts()

Department
HR            15
Operations    14
Marketing     12
Finance       11
Sales          8
IT             6
Name: count, dtype: int64

## 8. Standardize City

In [11]:
CITY_MAP = {"bangalore": "Bengaluru"}
df['City'] = (df['City'].astype(str).str.strip().str.lower()
              .replace(CITY_MAP).str.title())
df['City'] = df['City'].replace('Nan', np.nan)
df['City'].value_counts()

City
Pune         11
Mumbai       10
Bengaluru     9
Surat         7
Hyderabad     7
Delhi         6
Kolkata       5
Ahmedabad     5
Chennai       4
Jaipur        3
Name: count, dtype: int64

## 9. Parse Join_Date — 6+ formats into one standard date

In [12]:
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%d-%b-%Y", "%Y/%m/%d", "%B %d, %Y", "%d-%m-%Y", "%m-%d-%Y"]

def clean_date(val):
    if pd.isna(val):
        return pd.NaT
    val = str(val).strip()
    for fmt in DATE_FORMATS:
        try:
            return pd.to_datetime(val, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df['Join_Date'] = df['Join_Date'].apply(clean_date)
df['Join_Date'].head()

0   2022-07-19
1   2020-11-02
2   2020-11-02
3   2020-11-02
4   2021-05-14
Name: Join_Date, dtype: datetime64[us]

## 10. Clean Salary — strip currency symbols and commas

In [13]:
def clean_salary(val):
    if pd.isna(val):
        return np.nan
    val = re.sub(r"[₹$,]", "", str(val).strip())
    try:
        return float(val)
    except ValueError:
        return np.nan

df['Salary'] = df['Salary'].apply(clean_salary)
df['Salary'].describe()

count        52.000000
mean      73720.653846
std       28315.737268
min       26504.000000
25%       49639.250000
50%       77350.000000
75%      100392.000000
max      114889.000000
Name: Salary, dtype: float64

## 11. Clean Email and Rating, standardize Active

In [14]:
df['Email'] = df['Email'].astype(str).str.strip().str.lower().replace('nan', np.nan)

RATING_WORD_MAP = {"five": 5}
def clean_rating(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().lower()
    if val in RATING_WORD_MAP:
        return RATING_WORD_MAP[val]
    try:
        num = float(val)
    except ValueError:
        return np.nan
    if num < 1 or num > 5:
        return np.nan
    return num
df['Rating'] = df['Rating'].apply(clean_rating)

ACTIVE_MAP = {"yes": True, "y": True, "1": True, "true": True,
              "no": False, "n": False, "0": False, "false": False}
df['Active'] = df['Active'].astype(str).str.strip().str.lower().map(ACTIVE_MAP)

## 12. Remove duplicates

In [15]:
before = len(df)
df.drop_duplicates(inplace=True)
df.drop_duplicates(subset='EmployeeID', keep='first', inplace=True)
print(f"Removed {before - len(df)} duplicate rows")

Removed 9 duplicate rows


## 13. Final check and save

In [16]:
df.isna().sum()

EmployeeID     0
Full Name      0
Age           25
Gender         1
Department     3
City           2
Join_Date      4
Salary        16
Email          7
Rating        16
Active         5
dtype: int64

In [17]:
df.sort_values('EmployeeID', inplace=True)
df.to_csv('data/cleaned_data.csv', index=False)
print(f"Final cleaned shape: {df.shape}")
df.head(10)

Final cleaned shape: (60, 11)


,EmployeeID,Full Name,Age,Gender,Department,City,Join_Date,Salary,Email,Rating,Active
46,E1001,Aditya Verma,38.0,Female,Operations,Surat,2021-05-14,37280.0,aditya.verma78@gmail.com,5.0,False
33,E1002,N. Kumar,47.0,Male,Sales,Bengaluru,2021-05-14,61421.0,nikhil.kumar98@company.com,2.0,False
29,E1003,Sanjay Verma,NaN,Male,Finance,Hyderabad,2021-05-14,107397.0,sanjay.verma91@gmail.com,1.0,NaN
21,E1004,Neha Chopra,NaN,Female,Finance,Chennai,2021-05-14,52460.0,neha.chopra88@gmail.com,NaN,NaN
59,E1005,Anjali Joshi,30.0,Male,IT,Mumbai,2021-05-14,NaN,anjali.joshi28@gmail.com,4.5,False
47,E1006,Aditya Iyer,50.0,Female,Finance,Surat,2019-01-30,77350.0,aditya.iyer18@gmail.com,NaN,False
36,E1007,Suresh Sharma,NaN,Male,IT,Ahmedabad,2019-01-30,26504.0,suresh.sharma69@company.com,NaN,False
61,E1008,Sneha Singh,25.0,Male,Operations,Delhi,2021-05-14,NaN,sneha.singh20@company.com,3.0,True
12,E1009,S. Rao,NaN,Male,IT,Pune,2021-05-14,99364.0,suresh.rao63@gmail.com,5.0,True
58,E1010,Vikram Kumar,56.0,Female,Marketing,Bengaluru,2020-11-02,113039.0,vikram.kumar67@outlook.com,2.0,False


## Summary
- Standardized all missing-value markers into real `NaN`
- Removed duplicate rows and conflicting duplicate EmployeeIDs
- Fixed data types: Age, Salary, Rating → numeric; Join_Date → datetime
- Standardized inconsistent values across Gender, Department, City, Active